In [1]:
import os
from glob import glob

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.models import densenet121

__all__ = [
    "alexnet",
    "densenet121",
    "densenet169",
    "densenet201",
    "densenet161",
    "resnet18",
    "resnet34",
    "resnet50",
    "resnet101",
    "resnet152",
    "inceptionv3",
    "squeezenet1_0",
    "squeezenet1_1",
    "vgg11",
    "vgg11_bn",
    "vgg13",
    "vgg13_bn",
    "vgg16",
    "vgg16_bn",
    "vgg19_bn",
    "vgg19",
]

model_urls = {
    "alexnet": "https://download.pytorch.org/models/alexnet-owt-4df8aa71.pth",
    "densenet121": "http://data.lip6.fr/cadene/pretrainedmodels/densenet121-fbdb23505.pth",
    "densenet169": "http://data.lip6.fr/cadene/pretrainedmodels/densenet169-f470b90a4.pth",
    "densenet201": "http://data.lip6.fr/cadene/pretrainedmodels/densenet201-5750cbb1e.pth",
    "densenet161": "http://data.lip6.fr/cadene/pretrainedmodels/densenet161-347e6b360.pth",
    "inceptionv3": "https://download.pytorch.org/models/inception_v3_google-1a9a5a14.pth",
    "resnet18": "https://download.pytorch.org/models/resnet18-5c106cde.pth",
    "resnet34": "https://download.pytorch.org/models/resnet34-333f7ec4.pth",
    "resnet50": "https://download.pytorch.org/models/resnet50-19c8e357.pth",
    "resnet101": "https://download.pytorch.org/models/resnet101-5d3b4d8f.pth",
    "resnet152": "https://download.pytorch.org/models/resnet152-b121ed2d.pth",
    "squeezenet1_0": "https://download.pytorch.org/models/squeezenet1_0-a815701f.pth",
    "squeezenet1_1": "https://download.pytorch.org/models/squeezenet1_1-f364aa15.pth",
    "vgg11": "https://download.pytorch.org/models/vgg11-bbd30ac9.pth",
    "vgg13": "https://download.pytorch.org/models/vgg13-c768596a.pth",
    "vgg16": "https://download.pytorch.org/models/vgg16-397923af.pth",
    "vgg19": "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth",
    "vgg11_bn": "https://download.pytorch.org/models/vgg11_bn-6002323d.pth",
    "vgg13_bn": "https://download.pytorch.org/models/vgg13_bn-abd245e5.pth",
    "vgg16_bn": "https://download.pytorch.org/models/vgg16_bn-6c64b313.pth",
    "vvg19_bn": "https://download.pytorch.org/models/vgg19_bn-c79401a0.pth",
}




In [2]:
class GeM(nn.Module):
    """Generalized Mean (GeM) pooling layer."""

    def __init__(self, p=3, eps=1e-6):
        super(GeM, self).__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return gem(x, p=self.p, eps=self.eps)

    def __repr__(self):
        return f"{self.__class__.__name__}(p={self.p.item():.4f}, eps={self.eps})"


def gem(x, p=3, eps=1e-6):
    """Helper for GeM pooling."""
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p)


def get_densenet121_gem(pretrain="imagenet"):
    """
    Returns a DenseNet‑121 model with GeM pooling and a single linear output.
    If `pretrain` is "imagenet" we load ImageNet weights; otherwise the model
    is left uninitialised.
    """
    if pretrain == "imagenet":
        model = densenet121(pretrained=True)
    else:
        model = densenet121(pretrained=False)

    model.features.avg_pool = GeM()
    model.classifier = nn.Linear(1024, 1)
    return model




In [3]:
device = torch.device("cpu")

possible_paths = [
    "/kaggle/input/aptos2019-blindness-detection/test_images",
    "/kaggle/input/test_images",
    "./data/aptos2019-blindness-detection/test_images",
    "./input/aptos2019-blindness-detection/test_images",
    "./data/test_images",
    "./input/test_images",
]
TEST_IMAGE_PATH = None
for p in possible_paths:
    if os.path.isdir(p):
        TEST_IMAGE_PATH = p
        break
if TEST_IMAGE_PATH is None:
    print("Warning: Test image directory not found – will fall back to CSV IDs.")
    test_images = []
else:
    test_images = glob(os.path.join(TEST_IMAGE_PATH, "*.png"))


def _load_test_ids():
    """Read the test CSV and return a list of image IDs."""
    possible_test_paths = [
        "./test.csv",
        "./data/test.csv",
        "./input/test.csv",
        "/kaggle/input/aptos2019-blindness-detection/test.csv",
        "/kaggle/input/test.csv",
    ]
    for pt in possible_test_paths:
        if os.path.isfile(pt):
            df = pd.read_csv(pt)
            return df["id_code"].tolist()
    raise FileNotFoundError("Test CSV not found.")




In [4]:
def make_predictions(
    model, test_images, transform, size=256, device=torch.device("cpu")
):
    """
    Run inference on each image, apply horizontal flip augmentation,
    average the sigmoid outputs and map them to the 0‑4 diagnosis range.
    """
    model.eval()
    predictions = []
    for im_path in test_images:
        image = Image.open(im_path).convert("RGB")
        image = image.resize((size, size), resample=Image.BILINEAR)

        tensor = transform(image).unsqueeze(0).to(device)

        with torch.no_grad():
            out1 = model(tensor)
            out2 = model(torch.flip(tensor, dims=(3,)))  # horizontal flip
            pred1 = torch.sigmoid(out1).item()
            pred2 = torch.sigmoid(out2).item()
            pred = (pred1 + pred2) / 2.0 * 4.0  # average and map to 0‑4

        img_id = os.path.splitext(os.path.basename(im_path))[0]
        predictions.append((img_id, pred))
    return predictions


def _load_train_stats():
    """Return (mean, std) of the training diagnosis column."""
    possible_train_paths = [
        "./train.csv",
        "./data/train.csv",
        "./input/train.csv",
        "/kaggle/input/aptos2019-blindness-detection/train.csv",
        "/kaggle/input/train.csv",
    ]
    for pt in possible_train_paths:
        if os.path.isfile(pt):
            df = pd.read_csv(pt)
            mean = df["diagnosis"].mean()
            std = df["diagnosis"].std(ddof=0)
            return mean, max(std, 1e-3)  # avoid division by zero
    return 2.0, 1.0


def _load_train_mean():
    """Legacy helper – returns rounded mean (used for constant fallback)."""
    mean, _ = _load_train_stats()
    return int(round(mean))


try:
    model = get_densenet121_gem(pretrain="imagenet")
    print("Model loaded with ImageNet weights.")
except Exception as e:
    print(f"ImageNet weight loading failed ({e}), using untrained model.")
    model = get_densenet121_gem(pretrain=None)

model.to(device)

checkpoint_candidates = []
search_dirs = [
    "./",
    "./input",
    "./data",
    "/kaggle/input",
    "/kaggle/working",
]
for base in search_dirs:
    if os.path.isdir(base):
        checkpoint_candidates.extend(
            glob(os.path.join(base, "**", "*densenet121*.pth"), recursive=True)
        )
        checkpoint_candidates.extend(
            glob(os.path.join(base, "**", "*.pth"), recursive=True)
        )

custom_weight_path = None
if checkpoint_candidates:
    explicit_path = "../input/densenet121/model_densenet121_bs64_30.pth"
    if os.path.exists(explicit_path):
        custom_weight_path = explicit_path
    else:
        custom_weight_path = checkpoint_candidates[0]

if custom_weight_path and os.path.exists(custom_weight_path):
    try:
        state = torch.load(custom_weight_path, map_location=device)
        model.load_state_dict(state, strict=False)
        print(f"Custom weights loaded from {custom_weight_path}.")
    except Exception as e:
        print(
            f"Warning: could not load custom weights from {custom_weight_path} ({e}); using current model weights."
        )
else:
    print("No custom weight file found; using current model weights.")

norm = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

predictions = make_predictions(model, test_images, norm, size=224, device=device)

if len(predictions) == 0:
    # No image files – fall back to a simple mean‑class baseline.
    test_ids = _load_test_ids()
    mean_diag = _load_train_mean()
    predictions = [(tid, float(mean_diag)) for tid in test_ids]
    print(
        f"No image files found – generated baseline predictions (class {mean_diag}) for {len(test_ids)} test IDs."
    )
else:
    # Convert list of tuples to NumPy for easy statistics.
    pred_vals = np.array([p[1] for p in predictions])
    if pred_vals.std() < 1e-3:
        # Near‑constant predictions are replaced by the overall training mean.
        mean_diag = _load_train_mean()
        predictions = [(img_id, float(mean_diag)) for img_id, _ in predictions]
        print(
            f"Predictions were near‑constant; replaced with baseline class {mean_diag}."
        )
    else:
        # **Simplified post‑processing:** keep the sigmoid‑scaled 0‑4 values directly.
        # This removes the previous linear calibration that could distort the raw output.
        predictions = [
            (pid, float(val)) for (pid, _), val in zip(predictions, pred_vals)
        ]




/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100%|██████████| 30.8M/30.8M [00:00<00:00, 107MB/s]


Model loaded with ImageNet weights.
No custom weight file found; using current model weights.


In [5]:
pred_df = pd.DataFrame(predictions, columns=["id_code", "diagnosis"])
pred_df["diagnosis"] = pred_df["diagnosis"].round().astype(int)
pred_df["diagnosis"] = pred_df["diagnosis"].clip(0, 4)

submission_path = "submission.csv"
pred_df.to_csv(submission_path, index=False)
print(f"Submission file written to {submission_path}")
print(pred_df.head())

Submission file written to submission.csv
        id_code  diagnosis
0  218c822a3dd9          2
1  0e82bcacc475          1
2  683023cda6a5          2
3  81704925f759          1
4  26cd40b57ad1          2
